# Aneurysm Volume Prediction v2 Plus Tuned (from 0.7976 baseline)

在原始 `v2_attention_plus` 基础上的低风险提分改动：
- 按fold验证分数加权模型集成（替代简单平均）
- OOF两阶段搜索（粗网格 + 细网格）
- 保留原有效策略：attention + TTA + 分层CV + OOF校准
- 保持竞赛平台路径强适配


In [ ]:
# 如缺包请取消注释
# %pip install nibabel scikit-learn tqdm pandas matplotlib
# %pip install torch torchvision torchaudio


In [ ]:
import os
import sys
import json
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from scipy import ndimage

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = (DEVICE.type == "cuda")
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Device:", DEVICE, "AMP:", USE_AMP)


In [ ]:
# ===== 路径自动适配 =====
DATA_ROOT = os.environ.get("ANEURYSM_DATA_ROOT", "").strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / "train").exists() and (p / "test").exists() and (p / "train_labels").exists()

def find_base_dir():
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p

    fixed = [
        "/dataset/public",
        "/dataset",
        "/kaggle/input/aneurysm-volume-prediction",
        "/kaggle/input/aneurysm-volume",
        "/Users/songling/Desktop/Aneurysm Volume Prediction",
    ]
    for x in fixed:
        p = Path(x)
        if is_valid_dataset_dir(p):
            return p

    for root in [Path("/dataset"), Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d
    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    raise FileNotFoundError("未找到数据目录，请设置 ANEURYSM_DATA_ROOT")

TRAIN_IMG_DIR = BASE_DIR / "train"
TRAIN_MASK_DIR = BASE_DIR / "train_labels"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV = BASE_DIR / "train.csv"

if Path("/root/setup/solution/working").exists():
    OUTPUT_ROOT = Path("/root/setup/solution/working")
elif Path("/working").exists():
    OUTPUT_ROOT = Path("/working")
elif Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    OUTPUT_ROOT = BASE_DIR / "working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RUN_NAME", "").strip() or datetime.now().strftime("v2_attn_%Y%m%d_%H%M%S")
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / "checkpoints"
PRED_DIR = EXP_DIR / "predictions"
LOG_DIR = EXP_DIR / "logs"
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXP_DIR:", EXP_DIR)


In [ ]:
def load_nii(path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def robust_zscore(x, eps=1e-6):
    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    m, s = x.mean(), x.std()
    return (x - m) / (s + eps)

def volume_from_binary_mask(mask_zyx, spacing):
    voxel_vol = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(mask_zyx.sum() * voxel_vol, 0.0))

def volume_from_prob(prob_zyx, spacing):
    voxel_vol = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(prob_zyx.sum() * voxel_vol, 0.0))

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)))

def parse_pid(path: Path):
    return int(path.name.split(".")[0])


In [ ]:
def safe_percentile(arr, q):
    if arr.size == 0:
        return 0.0
    return float(np.percentile(arr, q))

def postprocess_mask(prob, thr=0.5, min_vox=0, keep_largest=True, conf_pctl=95.0, conf_thr=0.0):
    m = (prob > thr).astype(np.uint8)
    if m.sum() == 0:
        return m
    lbl, n = ndimage.label(m)
    if n <= 1 and min_vox <= 0 and conf_thr <= 0:
        return m

    keep = []
    for cid in range(1, n+1):
        comp = (lbl == cid)
        sz = int(comp.sum())
        cval = safe_percentile(prob[comp], conf_pctl)
        if sz >= int(min_vox) and cval >= float(conf_thr):
            keep.append((cid, sz, cval))

    if len(keep) == 0:
        # fallback to largest component
        sizes = [(cid, int((lbl == cid).sum())) for cid in range(1, n+1)]
        cid = sorted(sizes, key=lambda x: x[1], reverse=True)[0][0]
        return (lbl == cid).astype(np.uint8)

    if keep_largest:
        cid = sorted(keep, key=lambda x: x[1], reverse=True)[0][0]
        return (lbl == cid).astype(np.uint8)

    out = np.zeros_like(m, dtype=np.uint8)
    for cid, _, _ in keep:
        out[lbl == cid] = 1
    return out


In [ ]:
class AneurysmDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def _aug(self, img, msk):
        # (1,D,H,W)
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
            if msk is not None:
                msk = torch.flip(msk, dims=[2])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[3])
            if msk is not None:
                msk = torch.flip(msk, dims=[3])
        if random.random() < 0.5:
            k = random.randint(0, 3)
            img = torch.rot90(img, k=k, dims=[2, 3])
            if msk is not None:
                msk = torch.rot90(msk, k=k, dims=[2, 3])
        if random.random() < 0.7:
            scale = 1.0 + random.uniform(-0.15, 0.15)
            shift = random.uniform(-0.12, 0.12)
            img = img * scale + shift
        if random.random() < 0.3:
            img = img + torch.randn_like(img) * 0.03
        return img, msk

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        arr, spacing = load_nii(p)
        arr = robust_zscore(arr)
        arr = np.transpose(arr, (2, 0, 1)).astype(np.float32)  # (D,H,W)
        img = torch.from_numpy(arr).unsqueeze(0)

        out = {"image": img, "spacing": torch.tensor(spacing, dtype=torch.float32), "pid": parse_pid(p)}

        if self.mask_paths is not None:
            m, _ = load_nii(self.mask_paths[idx])
            m = (m > 0.5).astype(np.float32)
            m = np.transpose(m, (2, 0, 1)).astype(np.float32)
            msk = torch.from_numpy(m).unsqueeze(0)
            if self.augment:
                img, msk = self._aug(img, msk)
                out["image"] = img
            out["mask"] = msk
        return out


In [ ]:
def gn(ch):
    g = 8
    while ch % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, ch)

class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(ch // r, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(ch, hid, 1),
            nn.ReLU(inplace=True),
            nn.Conv3d(hid, ch, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.fc(self.pool(x))
        return x * w

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.n1 = gn(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.n2 = gn(out_ch)
        self.se = SEBlock3D(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        identity = self.proj(x)
        x = self.act(self.n1(self.conv1(x)))
        x = self.n2(self.conv2(x))
        x = self.se(x)
        x = self.act(x + identity)
        return x

class AttnGate3D(nn.Module):
    def __init__(self, x_ch, g_ch, inter_ch):
        super().__init__()
        self.wx = nn.Conv3d(x_ch, inter_ch, 1, bias=False)
        self.wg = nn.Conv3d(g_ch, inter_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv3d(inter_ch, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, x, g):
        a = self.psi(self.wx(x) + self.wg(g))
        return x * a

class AttentionUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=24):
        super().__init__()
        self.e1 = ConvBlock3D(in_ch, base)
        self.p1 = nn.MaxPool3d(2)
        self.e2 = ConvBlock3D(base, base*2)
        self.p2 = nn.MaxPool3d(2)
        self.e3 = ConvBlock3D(base*2, base*4)
        self.p3 = nn.MaxPool3d(2)
        self.b = ConvBlock3D(base*4, base*8)

        self.u3 = nn.ConvTranspose3d(base*8, base*4, 2, 2)
        self.a3 = AttnGate3D(base*4, base*4, base*2)
        self.d3 = ConvBlock3D(base*8, base*4)
        self.u2 = nn.ConvTranspose3d(base*4, base*2, 2, 2)
        self.a2 = AttnGate3D(base*2, base*2, base)
        self.d2 = ConvBlock3D(base*4, base*2)
        self.u1 = nn.ConvTranspose3d(base*2, base, 2, 2)
        self.a1 = AttnGate3D(base, base, max(base//2, 4))
        self.d1 = ConvBlock3D(base*2, base)
        self.head = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))

        d3 = self.u3(b)
        s3 = self.a3(e3, d3)
        d3 = self.d3(torch.cat([d3, s3], dim=1))
        d2 = self.u2(d3)
        s2 = self.a2(e2, d2)
        d2 = self.d2(torch.cat([d2, s2], dim=1))
        d1 = self.u1(d2)
        s1 = self.a1(e1, d1)
        d1 = self.d1(torch.cat([d1, s1], dim=1))
        return self.head(d1)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=4.5, bce_weight=0.45, smooth=1e-5):
        super().__init__()
        self.pos_weight = pos_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
    def forward(self, logits, target):
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
        p = torch.sigmoid(logits).reshape(logits.size(0), -1)
        t = target.reshape(target.size(0), -1)
        inter = (p*t).sum(dim=1)
        den = p.sum(dim=1) + t.sum(dim=1)
        dice = 1.0 - (2.0*inter + self.smooth) / (den + self.smooth)
        dice = dice.mean()
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["patient_id"] = train_df["patient_id"].astype(int)
all_train_imgs = sorted(TRAIN_IMG_DIR.glob("*.nii.gz"))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]
all_test_imgs = sorted(TEST_IMG_DIR.glob("*.nii.gz"))
print("train:", len(all_train_imgs), "test:", len(all_test_imgs))


In [ ]:
CFG = {
    "n_splits": 5,
    "epochs": 60,
    "batch_size": 2,
    "num_workers": 0,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "base_ch": 24,
    "grad_clip": 1.0,
    "tta": True,
    "early_stop_patience": 12,
    "seeds": [42, 2025],
    "fold_weight_power": 2.0,
}
print(CFG)


In [ ]:
def infer_prob_tta(model, x):
    # x: (1,1,D,H,W)
    out = []
    with torch.no_grad():
        p = torch.sigmoid(model(x)); out.append(p)
        p = torch.sigmoid(model(torch.flip(x, dims=[3]))); out.append(torch.flip(p, dims=[3]))
        p = torch.sigmoid(model(torch.flip(x, dims=[4]))); out.append(torch.flip(p, dims=[4]))
        p = torch.sigmoid(model(torch.flip(x, dims=[3,4]))); out.append(torch.flip(p, dims=[3,4]))
    return torch.mean(torch.stack(out, dim=0), dim=0)

def make_splits(train_df, n_splits=5, seed=42):
    vols = train_df.sort_values("patient_id")["volume"].values
    try:
        bins = pd.qcut(vols, q=5, labels=False, duplicates="drop")
    except Exception:
        bins = pd.cut(vols, bins=5, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.arange(len(all_train_imgs)), bins))

def train_one_fold(seed, fold, tr_idx, va_idx):
    seed_everything(seed + fold)
    tr_imgs = [all_train_imgs[i] for i in tr_idx]
    tr_msks = [all_train_msks[i] for i in tr_idx]
    va_imgs = [all_train_imgs[i] for i in va_idx]
    va_msks = [all_train_msks[i] for i in va_idx]

    tr_ds = AneurysmDataset(tr_imgs, tr_msks, augment=True)
    va_ds = AneurysmDataset(va_imgs, va_msks, augment=False)
    tr_ld = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])
    va_ld = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

    model = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
    crit = DiceBCELoss(pos_weight=4.5, bce_weight=0.45)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    sd = CKPT_DIR / f"seed_{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    ckpt = sd / f"fold_{fold}.pt"
    best_vs, no_improve = -1.0, 0

    for ep in range(CFG["epochs"]):
        model.train()
        losses = []
        for b in tr_ld:
            x = b["image"].to(DEVICE)
            y = b["mask"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logit = model(x)
                loss = crit(logit, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
        sch.step()

        model.eval()
        gts, prs = [], []
        with torch.no_grad():
            for b in va_ld:
                x = b["image"].to(DEVICE)
                m = b["mask"].cpu().numpy()[0,0]
                sp = b["spacing"].numpy()[0]
                p = torch.sigmoid(model(x)).cpu().numpy()[0,0]
                pm = (p > 0.5).astype(np.uint8)
                gts.append(volume_from_binary_mask(m, sp))
                prs.append(volume_from_binary_mask(pm, sp))
        vs = volumetric_similarity(gts, prs)
        print(f"Seed {seed} Fold {fold} Ep {ep+1:02d}/{CFG["epochs"]} loss={np.mean(losses):.4f} valVS={vs:.4f}")
        if vs > best_vs:
            best_vs = vs
            no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1
        if no_improve >= CFG["early_stop_patience"]:
            print(f"Seed {seed} Fold {fold} early stop. bestVS={best_vs:.4f}")
            break

    return ckpt, best_vs

def build_oof_items(model_paths_by_seed, splits, model_weights_by_seed_fold):
    oof = []
    for fold, (_, va_idx) in enumerate(splits):
        fold_models = []
        fold_weights = []
        for sd, paths in model_paths_by_seed.items():
            m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
            m.load_state_dict(torch.load(paths[fold], map_location=DEVICE))
            m.eval()
            fold_models.append(m)
            fold_weights.append(float(model_weights_by_seed_fold[(sd, fold)]))

        va_imgs = [all_train_imgs[i] for i in va_idx]
        va_msks = [all_train_msks[i] for i in va_idx]
        va_ds = AneurysmDataset(va_imgs, va_msks, augment=False)
        va_ld = DataLoader(va_ds, batch_size=1, shuffle=False)

        with torch.no_grad():
            for b in va_ld:
                pid = int(b["pid"][0]) if isinstance(b["pid"], torch.Tensor) else int(b["pid"])
                x = b["image"].to(DEVICE)
                gt_mask = b["mask"].cpu().numpy()[0,0].astype(np.uint8)
                sp = b["spacing"].numpy()[0].astype(np.float32)
                probs = []
                w = np.asarray(fold_weights, dtype=np.float64)
                w = w / (w.sum() + 1e-12)
                for m in fold_models:
                    if CFG["tta"]:
                        p = infer_prob_tta(m, x).cpu().numpy()[0,0]
                    else:
                        p = torch.sigmoid(m(x)).cpu().numpy()[0,0]
                    probs.append(p)
                prob = np.tensordot(w, np.stack(probs, axis=0), axes=(0,0)).astype(np.float16)
                oof.append({"pid": pid, "spacing": sp, "gt_mask": gt_mask, "prob": prob})
    return oof


In [ ]:
start = time.time()
splits = make_splits(train_df, n_splits=CFG["n_splits"], seed=42)
model_paths_by_seed = {}
model_weights_by_seed_fold = {}
score_rows = []

for seed in CFG["seeds"]:
    fold_paths = []
    for fold, (tr_idx, va_idx) in enumerate(splits):
        p, s = train_one_fold(seed, fold, tr_idx, va_idx)
        fold_paths.append(p)
        score_rows.append({"seed": seed, "fold": fold, "best_vs": float(s), "ckpt": str(p)})
        model_weights_by_seed_fold[(seed, fold)] = max(float(s), 1e-6)
    model_paths_by_seed[seed] = fold_paths

score_df = pd.DataFrame(score_rows)
score_df.to_csv(LOG_DIR / "fold_scores.csv", index=False)
print(score_df)
print(f"Training time: {(time.time()-start)/60:.1f} min")


In [ ]:
# ===== OOF调参：阈值 + 软硬融合 + 后处理 + 线性校准 =====
oof_items = build_oof_items(model_paths_by_seed, splits, model_weights_by_seed_fold)
gt_map = {int(r.patient_id): float(r.volume) for _, r in train_df.iterrows()}

# two-stage search: coarse -> fine + postprocess params
best = {"vs": -1.0, "thr": 0.5, "alpha": 1.0, "min_vox": 0, "keep_largest": True, "conf_thr": 0.0}

coarse_thr = np.arange(0.30, 0.66, 0.02)
coarse_alpha = np.arange(0.0, 1.01, 0.1)
coarse_min_vox = [0, 4, 8, 12]
coarse_keep_largest = [True, False]
coarse_conf_thr = [0.0, 0.35]

for thr in coarse_thr:
    for mv in coarse_min_vox:
        for kl in coarse_keep_largest:
            for ct in coarse_conf_thr:
                hard, soft, y = [], [], []
                for it in oof_items:
                    pid = int(it["pid"])
                    sp = it["spacing"]
                    prob = it["prob"].astype(np.float32)
                    pm = postprocess_mask(prob, thr=thr, min_vox=mv, keep_largest=kl, conf_pctl=95.0, conf_thr=ct)
                    hard.append(volume_from_binary_mask(pm, sp))
                    soft.append(volume_from_prob(prob, sp))
                    y.append(gt_map[pid])
                hard = np.asarray(hard, dtype=np.float64)
                soft = np.asarray(soft, dtype=np.float64)
                y = np.asarray(y, dtype=np.float64)
                for a in coarse_alpha:
                    pred = a * hard + (1.0 - a) * soft
                    vs = volumetric_similarity(y, pred)
                    if vs > best["vs"]:
                        best = {"vs": float(vs), "thr": float(thr), "alpha": float(a), "min_vox": int(mv), "keep_largest": bool(kl), "conf_thr": float(ct)}

fine_thr_l = max(0.05, best["thr"] - 0.03)
fine_thr_r = min(0.95, best["thr"] + 0.03)
fine_thr = np.arange(fine_thr_l, fine_thr_r + 1e-9, 0.005)
fine_alpha_l = max(0.0, best["alpha"] - 0.15)
fine_alpha_r = min(1.0, best["alpha"] + 0.15)
fine_alpha = np.arange(fine_alpha_l, fine_alpha_r + 1e-9, 0.05)
fine_min_vox = sorted(set([max(0, best["min_vox"]-4), best["min_vox"], best["min_vox"]+4]))
fine_keep_largest = [best["keep_largest"], not best["keep_largest"]]
fine_conf_thr = sorted(set([max(0.0, best["conf_thr"]-0.15), best["conf_thr"], min(0.9, best["conf_thr"]+0.15)]))

for thr in fine_thr:
    for mv in fine_min_vox:
        for kl in fine_keep_largest:
            for ct in fine_conf_thr:
                hard, soft, y = [], [], []
                for it in oof_items:
                    pid = int(it["pid"])
                    sp = it["spacing"]
                    prob = it["prob"].astype(np.float32)
                    pm = postprocess_mask(prob, thr=thr, min_vox=mv, keep_largest=kl, conf_pctl=95.0, conf_thr=ct)
                    hard.append(volume_from_binary_mask(pm, sp))
                    soft.append(volume_from_prob(prob, sp))
                    y.append(gt_map[pid])
                hard = np.asarray(hard, dtype=np.float64)
                soft = np.asarray(soft, dtype=np.float64)
                y = np.asarray(y, dtype=np.float64)
                for a in fine_alpha:
                    pred = a * hard + (1.0 - a) * soft
                    vs = volumetric_similarity(y, pred)
                    if vs > best["vs"]:
                        best = {"vs": float(vs), "thr": float(thr), "alpha": float(a), "min_vox": int(mv), "keep_largest": bool(kl), "conf_thr": float(ct)}

print("best oof before calib:", best)

oof_rows = []
for it in oof_items:
    pid = int(it["pid"])
    sp = it["spacing"]
    prob = it["prob"].astype(np.float32)
    pm = postprocess_mask(prob, thr=best["thr"], min_vox=best["min_vox"], keep_largest=best["keep_largest"], conf_pctl=95.0, conf_thr=best["conf_thr"])
    vh = volume_from_binary_mask(pm, sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    oof_rows.append({"patient_id": pid, "gt": gt_map[pid], "hard": vh, "soft": vs, "blend": blend})

oof_df = pd.DataFrame(oof_rows).sort_values("patient_id").reset_index(drop=True)
k, b = np.polyfit(oof_df["blend"].values, oof_df["gt"].values, deg=1)
oof_df["calibrated"] = np.clip(k * oof_df["blend"].values + b, 0, None)
vs_uncal = volumetric_similarity(oof_df["gt"].values, oof_df["blend"].values)
vs_cal = volumetric_similarity(oof_df["gt"].values, oof_df["calibrated"].values)
use_cal = bool(vs_cal >= vs_uncal)

print(f"OOF VS uncal: {vs_uncal:.6f}")
print(f"OOF VS cal  : {vs_cal:.6f}")
print("use_cal:", use_cal, "k=", round(float(k),6), "b=", round(float(b),6))

oof_df.to_csv(PRED_DIR / "oof_predictions.csv", index=False)


In [ ]:
# ===== Test推理 + 竞赛提交输出 =====
models = []
model_ws = []
for seed, plist in model_paths_by_seed.items():
    for fold, p in enumerate(plist):
        m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append((seed, fold, m))
        w = float(model_weights_by_seed_fold[(seed, fold)]) ** float(CFG.get("fold_weight_power", 1.0))
        model_ws.append(w)
model_ws = np.asarray(model_ws, dtype=np.float64)
model_ws = model_ws / (model_ws.sum() + 1e-12)

rows = []
for tp in tqdm(all_test_imgs, desc="Inference"):
    pid = parse_pid(tp)
    arr, sp = load_nii(tp)
    arr = robust_zscore(arr)
    arr = np.transpose(arr, (2,0,1)).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE)

    probs = []
    with torch.no_grad():
        for _,_,m in models:
            if CFG["tta"]:
                p = infer_prob_tta(m, x).cpu().numpy()[0,0]
            else:
                p = torch.sigmoid(m(x)).cpu().numpy()[0,0]
            probs.append(p)
    prob = np.tensordot(model_ws, np.stack(probs, axis=0), axes=(0,0))
    pm = postprocess_mask(prob, thr=best["thr"], min_vox=best["min_vox"], keep_largest=best["keep_largest"], conf_pctl=95.0, conf_thr=best["conf_thr"])
    vh = volume_from_binary_mask(pm, sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1.0 - best["alpha"]) * vs
    pred = np.clip(k * blend + b, 0, None) if use_cal else max(blend, 0.0)
    rows.append({"patient_id": pid, "volume": float(pred)})

sub = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
sub.to_csv(PRED_DIR / "submission.csv", index=False)

targets = [
    Path("/root/setup/solution/working/submission.csv"),
    Path("/working/submission.csv"),
    OUTPUT_ROOT / "submission.csv",
]
saved = []
for t in targets:
    try:
        t.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(t, index=False)
        saved.append(str(t))
    except Exception as e:
        print("skip", t, e)

meta = {
    "run_name": RUN_NAME,
    "cfg": CFG,
    "best": best,
    "calibration": {"use": use_cal, "k": float(k), "b": float(b), "oof_vs_uncal": float(vs_uncal), "oof_vs_cal": float(vs_cal)},
    "saved_submission_paths": saved,
}
with open(LOG_DIR / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved submission paths:")
for p in saved:
    print(" -", p)
print(sub.head())


In [ ]:
# 提交前自检
for p in [Path("/root/setup/solution/working/submission.csv"), Path("/working/submission.csv"), PRED_DIR / "submission.csv"]:
    print(p, "exists=", p.exists())
    if p.exists():
        d = pd.read_csv(p)
        print(" shape:", d.shape, " cols:", d.columns.tolist())
        print(d.head(3))


## 运行建议
- 先用默认配置跑一版（和0.7976版本差异最小）。
- 若时间允许，可把 `CFG["seeds"]` 改为 `[42, 2025, 3407]` 再试。
- 若过拟合迹象明显，先不要加大模型，优先减小 `epochs` 或增大 `early_stop_patience`。


## 参考思路
- Attention U-Net: [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)
- nnU-Net(自适应医学分割流程): [Nature Methods 2021](https://www.nature.com/articles/s41592-020-01008-z)
- TTA与连通域后处理在小病灶任务中常用于稳定体积估计。
